# Dia 16 — map() e funções para strings

**🃏 Cartas:** 16 · 37 · 58 &nbsp;·&nbsp; **📘 Original:** *Day 16. List mapping and functions for work with strings*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 16 — Os muitos nomes de Kau

Segunda-feira. O post-it de sexta continua colado no monitor, agora com mais uma linha escrita pela
Iara: **`Kaua T Brandao — KTB123`**.

— O RDAP te deu um nome sem acento e com o meio abreviado — ela diz, apoiando a caneca na sua mesa. —
O WhatsApp diz "Kau Ofertas". O Telegram, `kauofertas`. O e-mail, `kau.ofertas`. O registro que a
Delegada Rita mandou diz **Kauã Teixeira Brandão**. Quantas pessoas são essas?

Você arrisca: uma só?

— Pode ser. Mas o computador acha que são cinco. Para ele, "Brandão" e "brandao" são palavras
diferentes, e "Kaua T" não tem nada a ver com "Kauã Teixeira". Hoje você vai ensinar o computador a
enxergar como a gente.

Ela desenha no quadro uma seta saindo de um nome e se abrindo em dezenas de setas menores.

— E vai fazer o caminho inverso também: a partir do nome, **gerar** os usernames que ele pode ter
usado em outros lugares. Golpista reaproveita apelido. Sempre. A lista sai daqui e alimenta as
ferramentas do Dia 3.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-16-map-e-strings")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Entender `map()`: aplicar uma função a cada item de uma lista.
- Usar `title()`, `capitalize()`, `replace()` e `split()` como no curso original.
- Corrigir o `title()` para nomes brasileiros (da, de, dos, e).
- Remover acentos com `unicodedata` e criar uma "chave de comparação".
- Gerar variantes de username a partir de um nome e de um apelido.
- Desmontar uma linha de conversa exportada com `split`, `strip` e `join`.

## 🧠 Conceito em 2 minutos

**`map(função, lista)`** pega uma função e a aplica em **cada** item da lista, devolvendo os
resultados na mesma ordem. É o jeito curto de escrever um laço `for` que transforma tudo:

```python
list(map(str.upper, ["ana", "bia"]))   # ['ANA', 'BIA']
[n.upper() for n in ["ana", "bia"]]   # a mesma coisa, como "list comprehension"
```

Repare que a função vai **sem parênteses**: você entrega a função ao `map`, e ele a chama.
`map()` devolve um objeto "preguiçoso"; `list()` força o cálculo. Seu par é **`filter(função, lista)`**,
que fica só com os itens para os quais a função devolve `True`.

Os métodos de texto mais úteis em OSINT:

| Método | Exemplo | Resultado |
|---|---|---|
| `title()` | `"maria da silva".title()` | `'Maria Da Silva'` (erra o "da") |
| `capitalize()` | `"belo horizonte".capitalize()` | `'Belo horizonte'` |
| `replace(a, b)` | `"kau_ofertas".replace("_", ".")` | `'kau.ofertas'` |
| `split(sep, n)` | `"a:b:c".split(":", 1)` | `['a', 'b:c']` |
| `sep.join(lista)` | `".".join(["kau", "ofertas"])` | `'kau.ofertas'` |
| `strip()` | `"  oi \n".strip()` | `'oi'` |
| `casefold()` | `"BRANDÃO".casefold()` | `'brandão'` |

**Acentos** não se tiram com `replace` letra por letra: a forma normalizada **NFKD** do Unicode
separa "ã" em "a" + "~", e aí basta descartar os acentos soltos (`unicodedata.combining`).

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/array_map.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/original/array_map.py) | Dobra cada número de uma lista com `map()` |
| [`original/capallwords.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/original/capallwords.py) | Maiúscula em todas as palavras com `title()` |
| [`original/capfirstword.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/original/capfirstword.py) | Maiúscula só na primeira letra com `capitalize()` |
| [`original/replace.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/original/replace.py) | Troca `_` por `-` em várias palavras |
| [`original/split_string.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/original/split_string.py) | Quebra uma linha separada por `;` |

Nenhum precisa de `pip install`.

```bash
cd dia-16-map-e-strings
python original/array_map.py
python original/split_string.py
```

> 💡 Rode `split_string.py` e repare na **linha em branco** no fim: o `;` final gera um pedaço vazio.
> Em dados reais, isso vira uma coluna fantasma. Como você resolveria? (dica: `strip(";")` antes.)

O autor também cita o **chepy**, uma versão em Python do CyberChef (decodifica Base64, URL, binário
etc.). Vale conhecer: `pip install chepy`.

### `original/array_map.py` — Aplicar uma função a cada item de uma lista com map()

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Aplicar uma função a cada item de uma lista com map()
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_16/array_map.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────
# 💬 Uma função simples: recebe um número e devolve o dobro.
def doubleNumber(x):
  return x * 2

# 💬 A lista que vai passar pela função, item por item.
numbers = [2,4,8]

# 💬 map(função, lista) aplica a função em CADA item. Repare: passamos
# 💬 o NOME da função, sem parênteses (não estamos chamando, estamos entregando).
result = map(doubleNumber,numbers)

# 💬 map() devolve um objeto 'preguiçoso'; list() força o cálculo e mostra tudo.
# 💬 Saída esperada: [4, 8, 16]
print(list(result))

### `original/capallwords.py` — Primeira letra maiúscula em todas as palavras com title()

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Primeira letra maiúscula em todas as palavras com title()
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_16/capallwords.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────
# 💬 str.title() põe maiúscula no começo de CADA palavra.
def capitalizeAllWords(x):
  return x.title()

# 💬 Cidades dos EUA, todas em minúsculas.
cities = ["new york","los angeles","chicago"]

# 💬 Aplica a função em cada cidade.
result = map(capitalizeAllWords,cities)

# 💬 Saída: ['New York', 'Los Angeles', 'Chicago'].
# 💬 Em nomes brasileiros, title() erra: 'Maria Da Silva'. Veja br/01_nomes_br.py.
print(list(result))

### `original/capfirstword.py` — Só a primeira letra maiúscula com capitalize()

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Só a primeira letra maiúscula com capitalize()
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_16/capfirstword.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────
# 💬 str.capitalize() deixa maiúscula só a PRIMEIRA letra do texto todo
# 💬 (e põe o resto em minúsculas).
def capitalizeFirstWord(x):
  return x.capitalize()

cities = ["new york","los angeles","chicago"]

# 💬 Mesma lista, outra função: compare com capallwords.py.
result = map(capitalizeFirstWord,cities)

# 💬 Saída: ['New york', 'Los angeles', 'Chicago']
print(list(result))

### `original/replace.py` — Trocar caracteres em várias palavras com replace()

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Trocar caracteres em várias palavras com replace()
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_16/replace.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────
# 💬 str.replace(velho, novo) troca TODAS as ocorrências de '_' por '-'.
def replaceDash(x):
  return x.replace("_","-")

# 💬 Palavras no estilo snake_case (com sublinhado).
words = ["six_pack","king_size","editor_in_chief"]

# 💬 Em OSINT isso vira gerador de variantes de username:
# 💬 kau_ofertas → kau-ofertas. Veja br/03_variantes_username.py.
result = map(replaceDash,words)

# 💬 Saída: ['six-pack', 'king-size', 'editor-in-chief']
print(list(result))

### `original/split_string.py` — Quebrar um texto em pedaços com split()

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Quebrar um texto em pedaços com split()
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_16/split_string.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# ──────────────────────────────────────────────────────────────────────────
# 💬 Uma linha com campos separados por ';' (como um CSV).
# 💬 Repare no ';' do final: ele gera um último pedaço VAZIO.
string = "first name;last name;email;address;"

# 💬 split(';') corta o texto em cada ';' e devolve uma lista.
fields = string.split(";")

# 💬 Percorre a lista e imprime um campo por linha (a última linha sai em branco).
for field in fields:
    print (field)

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão** e 100% offline (usa os dados do caso).

| Arquivo | O que faz |
|---|---|
| [`br/01_nomes_br.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/br/01_nomes_br.py) | `nome_br()`: corrige o `title()` (da/de/dos/e minúsculas) e aplica com `map` nos nomes das vítimas |
| [`br/02_sem_acento.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/br/02_sem_acento.py) | Chave de comparação sem acento (NFKD + `casefold`); `map` + `filter`; o sobrenome do suspeito nas fontes |
| [`br/03_variantes_username.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/br/03_variantes_username.py) | Gera centenas de usernames prováveis de "Kauã Teixeira Brandão" e `kau.ofertas` → `saidas/variantes_username.txt` |
| [`br/04_split_e_join.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-16-map-e-strings/br/04_split_e_join.py) | Desmonta `[data hora] autor: mensagem` da conversa do WhatsApp e grava `saidas/conversa_estruturada.csv` |

```bash
cd dia-16-map-e-strings
python br/01_nomes_br.py
python br/03_variantes_username.py
```

**O que você deve ver no caso:** RDAP e WHOIS trazem `Kaua T Brandao`, que vira a mesma chave que
`Kauã Teixeira Brandão` quando você tira os acentos e aceita a inicial do meio. As iniciais **ktb**
batem com o handle `KTB123` do registro. E o sobrenome Brandão reaparece no titular de uma das contas
laranja (`M. R. Brandão Comércio ME`): parentesco? coincidência? Anote como **hipótese**.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_nomes_br.py` — Nomes brasileiros com maiúsculas certas: o title() erra, a gente conserta.

In [ ]:
%cd -q {PASTA_NB}
"""
Nomes brasileiros com maiúsculas certas: o title() erra, a gente conserta.

Carta 37 (rank A) — Jutsu dos Muitos Nomes

O que você aprende:
  • por que "maria da silva".title() vira "Maria Da Silva" (e por que isso atrapalha);
  • a escrever uma função que deixa da/de/do/das/dos/e em minúsculas;
  • a aplicar essa função em uma lista inteira com map().

Como rodar (dentro da pasta dia-16-map-e-strings):
    python br/01_nomes_br.py
"""

import csv                  # ler a planilha de denúncias
from pathlib import Path    # montar caminhos de arquivo

# ── Parâmetros (no Colab viram um formulário) ─────────────────────────────
QUANTOS_MOSTRAR = 8  #@param {type:"integer"}

# ── Onde estão os dados do caso ───────────────────────────────────────────
PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

# Palavras que, no meio de um nome brasileiro, ficam em minúsculas.
PARTICULAS = {"da", "de", "do", "das", "dos", "e", "d'"}


def nome_br(texto):
    """Põe um nome no formato brasileiro.

    "MARIA DA SILVA" → "Maria da Silva" · "joão dos santos e souza" → "João dos Santos e Souza"
    """
    # 1. split() sem argumento quebra em QUALQUER quantidade de espaços e
    #    ignora espaços nas pontas: "  ana   lima " → ["ana", "lima"]
    palavras = texto.lower().split()
    resultado = []
    # 2. enumerate dá a posição (0, 1, 2...) junto com a palavra
    for posicao, palavra in enumerate(palavras):
        if posicao > 0 and palavra in PARTICULAS:
            resultado.append(palavra)               # partícula no meio: fica minúscula
        else:
            resultado.append(palavra.capitalize())  # "brandão" → "Brandão"
    # 3. join() faz o caminho inverso do split(): cola a lista com um espaço
    return " ".join(resultado)


# ── 1. O problema do title() ──────────────────────────────────────────────
exemplos = ["MARIA DA SILVA", "joão dos santos e souza", "  kauã   teixeira brandão ", "ANA DE JESUS"]
print("🔤 title() do Python  ×  nome_br() desta carta\n")
for exemplo in exemplos:
    print(f"   {exemplo.strip().title():<28} → {nome_br(exemplo)}")

# ── 2. Lendo os nomes das vítimas ─────────────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    nomes = [linha["nome_vitima"] for linha in csv.DictReader(arquivo, delimiter=";")]

# Simulamos o que acontece de verdade: cada sistema grava de um jeito.
# map(str.upper, nomes) aplica o método upper() em cada nome (tudo MAIÚSCULO).
gritados = list(map(str.upper, nomes))

# ── 3. A mágica: map() aplica nome_br em cada item ────────────────────────
# Repare: passamos nome_br SEM parênteses. Quem chama a função é o map().
consertados = list(map(nome_br, gritados))

print(f"\n📋 {len(nomes)} nomes de vítimas: do sistema que grita → padrão brasileiro\n")
for antes, depois in zip(gritados[:QUANTOS_MOSTRAR], consertados[:QUANTOS_MOSTRAR]):
    print(f"   {antes:<30} → {depois}")

# ── 4. Conferência: voltamos ao original? ─────────────────────────────────
# zip() junta duas listas lado a lado; contamos quantos pares são iguais.
iguais = sum(1 for original, refeito in zip(nomes, consertados) if original == refeito)
print(f"\n✅ {iguais} de {len(nomes)} nomes voltaram exatamente como estavam na planilha.")

# ── 5. Por que isso importa em OSINT ──────────────────────────────────────
# Em buscas, "Kauã Teixeira Brandão" e "KAUÃ TEIXEIRA BRANDÃO" são o mesmo texto
# para um humano, mas NÃO para o computador. Padronizar antes de comparar evita
# perder um vínculo (falso negativo) ou duplicar uma pessoa no relatório.
print("\n💡 Padronize ANTES de comparar ou deduplicar. Amanhã esses nomes vão para o relatório.")

### `br/02_sem_acento.py` — Sem acento, sem maiúscula: a "chave de comparação" que junta grafias diferentes.

In [ ]:
%cd -q {PASTA_NB}
"""
Sem acento, sem maiúscula: a "chave de comparação" que junta grafias diferentes.

Carta 37 (rank A) — Jutsu dos Muitos Nomes (parte 2)

O que você aprende:
  • a tirar acentos com unicodedata (NFKD): "Kauã Brandão" → "kaua brandao";
  • a diferença entre lower() e casefold();
  • a combinar map() (transformar tudo) com filter() (ficar só com o que interessa).

Como rodar (dentro da pasta dia-16-map-e-strings):
    python br/02_sem_acento.py
"""

import csv
import json
import unicodedata          # tabela oficial Unicode: sabe que "ã" = "a" + "~"
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
PROCURAR = "brandao"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"


def sem_acento(texto):
    """Remove acentos e cedilha, mantendo as letras base.

    NFKD separa cada letra acentuada em duas peças: a letra e o acento.
    Depois jogamos fora as peças que são "marcas combinantes" (os acentos).
    """
    decomposto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in decomposto if not unicodedata.combining(c))


def chave(texto):
    """Chave de comparação: sem acento, minúscula 'agressiva' e espaços arrumados."""
    # casefold() é um lower() mais forte, pensado para comparar textos
    # (ex.: o "ß" alemão vira "ss"). Para nomes BR dá o mesmo, mas é o hábito certo.
    return " ".join(sem_acento(texto).casefold().split())


# ── 1. Demonstração ───────────────────────────────────────────────────────
print("🧪 Antes → chave de comparação\n")
for exemplo in ["Kauã Teixeira Brandão", "KAUA T. BRANDAO", "Poços de Caldas", "Pocos de Caldas", "  São  João "]:
    print(f"   {exemplo!r:<26} → {chave(exemplo)!r}")

# ── 2. Os nomes do suspeito espalhados pelas evidências ───────────────────
# Cada fonte grafou o nome de um jeito. Coletamos de onde eles aparecem.
rdap = json.loads((PASTA_CASO / "rdap_megaoferta.json").read_text(encoding="utf-8"))
nome_rdap = next(c[3] for ent in rdap["entities"] for c in ent["vcardArray"][1] if c[0] == "fn")

nome_whois = ""
for linha in (PASTA_CASO / "whois_megaoferta.txt").read_text(encoding="utf-8").splitlines():
    if linha.startswith("owner:"):
        nome_whois = linha.split(":", 1)[1].strip()   # tudo depois do primeiro ":"

nome_painel = ""
for linha in (PASTA_CASO / "painel_dump.sql").read_text(encoding="utf-8").splitlines():
    if "Brandão" in linha and "contas_recebimento" in linha:
        nome_painel = linha.split("'")[1]             # o texto entre as primeiras aspas simples

fontes = {"RDAP": nome_rdap, "WHOIS": nome_whois, "dump do painel": nome_painel,
          "registro informado à polícia": "Kauã Teixeira Brandão"}

print("\n🕵️ O mesmo sobrenome em fontes diferentes:\n")
for fonte, nome in fontes.items():
    print(f"   {fonte:<30} {nome:<28} → {chave(nome)}")

# ── 3. map + filter nos nomes das vítimas ─────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    linhas = list(csv.DictReader(arquivo, delimiter=";"))

nomes = [l["nome_vitima"] for l in linhas]
chaves = list(map(chave, nomes))                             # transforma TODOS
alvo = chave(PROCURAR)
com_alvo = list(filter(lambda c: alvo in c, chaves))         # fica só com quem contém o alvo
# lambda é uma função de uma linha sem nome: "recebe c, devolve (alvo in c)".

print(f"\n🔎 Vítimas cujo nome contém '{alvo}': {len(com_alvo)}")
print("   (Zero é a resposta esperada: o sobrenome do suspeito não aparece entre as vítimas.)")

# ── 4. Bônus: cidades digitadas de jeitos diferentes ──────────────────────
cidades = [l["cidade"] for l in linhas]
print(f"\n🏙️ Cidades como as vítimas digitaram: {len(set(cidades))} grafias diferentes")
print(f"🏙️ Depois da chave sem acento:       {len(set(map(chave, cidades)))} grafias diferentes")
print("   (Os erros de digitação, como 'Belo Horizone', continuam: isso é trabalho do Dia 7, com difflib.)")

### `br/03_variantes_username.py` — Gerador de variantes de username: os muitos nomes de Kau.

In [ ]:
%cd -q {PASTA_NB}
"""
Gerador de variantes de username: os muitos nomes de Kau.

Carta 37 (rank A) — Jutsu dos Muitos Nomes (parte 3)

O que você aprende:
  • a montar usernames prováveis a partir de um nome completo e de um apelido;
  • separadores (. _ -), iniciais, sufixos numéricos e "leetspeak" simples;
  • a usar set() para não repetir e sorted() para ordenar;
  • a salvar a lista num arquivo para alimentar ferramentas como blackbird/maigret (Dia 3).

Como rodar (dentro da pasta dia-16-map-e-strings):
    python br/03_variantes_username.py
"""

import unicodedata
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
NOME_COMPLETO = "Kauã Teixeira Brandão"  #@param {type:"string"}
APELIDO = "kau.ofertas"  #@param {type:"string"}
SUFIXOS = "1, 01, 2025, 31, br, oficial"  #@param {type:"string"}
USAR_LEET = True  #@param {type:"boolean"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
SAIDAS = PASTA_DIA / "saidas"

SEPARADORES = ["", ".", "_", "-"]
# Leetspeak "leve": só as trocas mais comuns. Mais que isso explode a lista.
LEET = str.maketrans({"a": "4", "e": "3", "o": "0", "i": "1"})
PARTICULAS = {"da", "de", "do", "das", "dos", "e"}


def limpar(texto):
    """Sem acento, minúsculo e só letras/números (usernames não aceitam 'ã')."""
    sem = "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))
    return "".join(c for c in sem.lower() if c.isalnum())


# ── 1. Quebrar o nome em partes úteis ─────────────────────────────────────
partes = [limpar(p) for p in NOME_COMPLETO.split() if p.lower() not in PARTICULAS]
primeiro, ultimo = partes[0], partes[-1]           # "kaua", "brandao"
meio = partes[1:-1]                                # ["teixeira"]
print(f"🧩 Partes do nome: {partes}")

# ── 2. Combinações "base" (sem separador ainda) ───────────────────────────
# Cada item é uma lista de pedaços; o separador entra depois.
bases = [
    [primeiro, ultimo],                    # kaua brandao
    [ultimo, primeiro],                    # brandao kaua
    [primeiro[0], ultimo],                 # k brandao
    [primeiro, ultimo[0]],                 # kaua b
    [primeiro] + meio + [ultimo],          # kaua teixeira brandao
    ["".join(p[0] for p in partes)],       # ktb (as iniciais: lembra do handle KTB123 do RDAP?)
    [primeiro[:3], ultimo],                # kau brandao
]

# ── 3. O apelido já conhecido também gera variantes ───────────────────────
# "kau.ofertas" → pedaços ["kau", "ofertas"]: trocamos o separador.
pedacos_apelido = [p for p in APELIDO.replace("_", ".").replace("-", ".").split(".") if p]
bases.append(pedacos_apelido)
bases.append(list(reversed(pedacos_apelido)))      # ofertas.kau

# ── 4. Aplicar separadores ────────────────────────────────────────────────
variantes = set()                                  # set: repetidos somem sozinhos
for pedacos in bases:
    for sep in SEPARADORES:
        variantes.add(sep.join(pedacos))

# ── 5. Sufixos numéricos e leetspeak ──────────────────────────────────────
sufixos = [s.strip() for s in SUFIXOS.split(",") if s.strip()]
for v in list(variantes):                          # list(): congelamos antes de mexer no set
    for s in sufixos:
        variantes.add(v + s)
        variantes.add(f"{v}_{s}")
    if USAR_LEET:
        variantes.add(v.translate(LEET))           # "kau.ofertas" → "k4u.0f3rt4s"

# Usernames de 1 ou 2 letras não servem para nada: filtramos.
variantes = {v for v in variantes if len(v) >= 3}

# ── 6. Ordenar: primeiro as "limpas" (sem números), depois as mais curtas, depois A→Z
# A key devolve uma tupla; o Python compara o 1º item, depois o 2º, e assim por diante.
# any(c.isdigit() for c in v) é False (=0) para "kaua.brandao" e True (=1) para "kaua1".
ordenadas = sorted(variantes, key=lambda v: (any(c.isdigit() for c in v), len(v), v))
print(f"\n🧮 {len(ordenadas)} variantes geradas. As 20 primeiras:\n")
for v in ordenadas[:20]:
    print(f"   {v}")

# ── 7. Salvar (um por linha: formato que ferramentas de username aceitam) ─
SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "variantes_username.txt"
destino.write_text("\n".join(ordenadas) + "\n", encoding="utf-8")
print(f"\n💾 Lista completa em {destino.relative_to(PASTA_DIA)}")
print("⚠️ Variante NÃO é identidade: cada conta encontrada precisa ser confirmada por outros vínculos.")

### `br/04_split_e_join.py` — split, join e strip na prática: desmontar uma conversa exportada do WhatsApp.

In [ ]:
%cd -q {PASTA_NB}
"""
split, join e strip na prática: desmontar uma conversa exportada do WhatsApp.

Carta 37 (rank A) — Jutsu dos Muitos Nomes (parte 4)

O que você aprende:
  • a quebrar a linha "[28/11/2025 21:14] Kau Ofertas: mensagem" em data, hora, autor e texto;
  • a usar split com limite (maxsplit) para não cortar o ":" que aparece dentro da mensagem;
  • a limpar espaços com strip() e a remontar textos com join().

Como rodar (dentro da pasta dia-16-map-e-strings):
    python br/04_split_e_join.py
"""

import csv
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
AUTOR_FOCO = "Kau Ofertas"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"


def desmontar(linha):
    """'[28/11/2025 21:14] Kau Ofertas: Boa noite!' → dict com data, hora, autor, mensagem.

    Devolve None se a linha não estiver no formato esperado.
    """
    linha = linha.strip()                       # tira espaços e o "\n" das pontas
    if not linha.startswith("["):
        return None
    # 1. Separar o carimbo "[...]" do resto. split("]", 1) corta só no PRIMEIRO "]".
    carimbo, resto = linha[1:].split("]", 1)    # "28/11/2025 21:14" e " Kau Ofertas: Boa noite!"
    data, hora = carimbo.split()                # split() sem argumento: corta no espaço
    # 2. Autor e mensagem. maxsplit=1 é essencial: a mensagem pode ter ":"
    #    (ex.: "Chave PIX: f38b..."), e não queremos cortar ali.
    if ":" not in resto:
        return None
    autor, mensagem = resto.split(":", 1)
    return {"data": data, "hora": hora, "autor": autor.strip(), "mensagem": mensagem.strip()}


# ── 1. Um exemplo comentado ───────────────────────────────────────────────
exemplo = "[28/11/2025 21:16] Kau Ofertas: Chave PIX: f38b2ffc-80a4-df5a-51c9-bc701e7ea419"
# Armadilha 1: a hora "21:16" também tem ":". Por isso primeiro tiramos o carimbo "[...]".
resto = exemplo.split("]", 1)[1]
print(f"✂️  Depois do ']':          {resto!r}")
print("🔪 split(':') sem limite corta demais:")
print(f"   {resto.split(':')}")
print("✅ split(':', 1) corta só uma vez (autor | mensagem):")
print(f"   {resto.split(':', 1)}")
print(f"\n🧩 desmontar() →  {desmontar(exemplo)}")

# ── 2. A conversa inteira ─────────────────────────────────────────────────
texto = (PASTA_CASO / "conversa_whatsapp.txt").read_text(encoding="utf-8")
mensagens = [m for m in map(desmontar, texto.splitlines()) if m]   # map + descarta os None
print(f"\n💬 {len(mensagens)} mensagens lidas.")

# ── 3. Quem falou quanto? ─────────────────────────────────────────────────
contagem = {}
for m in mensagens:
    contagem[m["autor"]] = contagem.get(m["autor"], 0) + 1
for autor, total in contagem.items():
    print(f"   {autor:<14} {'▇' * total} {total}")

# ── 4. Só as falas do autor em foco, remontadas com join ──────────────────
falas = [f"{m['data']} {m['hora']} — {m['mensagem']}" for m in mensagens if m["autor"] == AUTOR_FOCO]
print(f"\n🗣️ Falas de {AUTOR_FOCO}:\n")
print("\n".join(f"   {f}" for f in falas))

# Palavras mais longas que ele usa: split() quebra a frase em palavras;
# strip(".,!?") tira a pontuação colada nas pontas de cada uma.
palavras = {p.strip(".,!?🔥").lower() for m in mensagens if m["autor"] == AUTOR_FOCO
            for p in m["mensagem"].split()}
print(f"\n🔤 Vocabulário de {AUTOR_FOCO}: {len(palavras)} palavras diferentes.")

# ── 5. Salvar como CSV (agora dá para abrir numa planilha) ────────────────
SAIDAS.mkdir(exist_ok=True)
with open(SAIDAS / "conversa_estruturada.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=["data", "hora", "autor", "mensagem"], delimiter=";")
    escritor.writeheader()
    escritor.writerows(mensagens)
print("💾 saidas/conversa_estruturada.csv gravado.")

## ⚖️ Ética, LGPD e OPSEC

- **Variante não é identidade.** Achar uma conta `kaua.brandao` em uma rede social não prova nada;
  existem homônimos. Só junte perfis com vínculos independentes (mesma foto, mesmo telefone, mesma
  chave PIX, horários coerentes) e registre cada vínculo.
- **Sobrenome igual não é parentesco.** No relatório, escreva "titular com o mesmo sobrenome",
  nunca "parente do suspeito", até que algo confirme.
- Testar centenas de usernames em sites de terceiros é **coleta automatizada**: respeite limites, não
  faça login com conta fantoche para isso e lembre que a requisição sai do seu IP (Dia 12).
- Ao padronizar nomes de vítimas, você está **tratando dados pessoais** (LGPD). Trabalhe numa cópia,
  nunca altere a planilha original da denúncia (Dia 14: cadeia de custódia).

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. O que list(map(str.upper, ['ana', 'bia'])) devolve?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) 'ANABIA'", "B) ['ANA', 'BIA']", "C) Um erro, porque faltam os parênteses em upper", "D) ['Ana', 'Bia']"]
correta = "B) ['ANA', 'BIA']"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "map aplica str.upper em cada item; a função vai sem parênteses porque quem a chama é o map.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Por que usar split(':', 1) para separar autor e mensagem em '[...] Kau: Chave PIX: f38b...'?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Porque é mais rápido", "B) Para cortar só no primeiro ':' e não quebrar a mensagem, que também tem ':'", "C) Porque split(':') não funciona com acentos", "D) Para remover os espaços"]
correta = "B) Para cortar só no primeiro ':' e não quebrar a mensagem, que também tem ':'"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O segundo argumento limita o número de cortes; o resto da linha fica inteiro.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Qual é o jeito certo de transformar 'Brandão' em 'brandao'?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) 'Brandão'.lower()", "B) 'Brandão'.replace('ã', 'a') para cada letra possível", "C) Normalizar com unicodedata NFKD, descartar os acentos e aplicar casefold()", "D) 'Brandão'.title()"]
correta = "C) Normalizar com unicodedata NFKD, descartar os acentos e aplicar casefold()"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "NFKD separa a letra do acento; descartando as marcas combinantes, sobra a letra base.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 16: Detector de grafias compatíveis

**Carta 58 · Rank S**

## A missão

O RDAP diz `Kaua T Brandao`. A Delegada Rita diz `Kauã Teixeira Brandão`. A Iara quer uma função
que responda, sem achismo: **essas duas grafias podem ser a mesma pessoa?**

Escreva um script que:

1. Tenha uma função `compativel(nome1, nome2)` que devolva `True` ou `False`.
2. Ignore acentos, maiúsculas, pontuação (`T.`) e partículas (da, de, dos, e).
3. Aceite **inicial no lugar do nome do meio** (`T` ≈ `Teixeira`), mas exija o **último sobrenome
   por extenso** (para não juntar "K. Brandão" com qualquer Brandão).
4. Recolha as grafias de nome que aparecem no RDAP, no WHOIS e em `denuncias.csv`, e **agrupe** as
   compatíveis entre si.
5. Liste os pares "quase iguais" que a regra **não** juntou, para revisão humana.

## Critérios de rank S

- [ ] Só biblioteca padrão, 100% offline
- [ ] `compativel("Kaua T Brandao", "Kauã Teixeira Brandão")` é `True`
- [ ] `compativel("M. R. Brandão", "Kauã Teixeira Brandão")` é `False`
- [ ] O script tem pelo menos 4 `assert` testando a regra
- [ ] Mostra o que precisa de revisão humana (a máquina sugere, você decide)

## Dica

Transforme cada nome numa lista de "tokens" limpos primeiro. Depois compare os tokens dois a dois com
`zip()`. Dois tokens combinam se forem iguais **ou** se um deles tiver uma letra só e for a inicial do
outro.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 16 — Detector de grafias compatíveis.

Decide se duas formas de um nome podem ser a MESMA pessoa, aceitando acentos,
maiúsculas, pontuação e iniciais abreviadas ("Kaua T Brandao" ≈ "Kauã Teixeira Brandão").
Depois agrupa todas as grafias encontradas nas evidências do caso.
100% offline: usa só caso/dados/.
"""

import csv
import itertools
import json
import unicodedata
from pathlib import Path

NOME_ALVO = "Kauã Teixeira Brandão"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
PARTICULAS = {"da", "de", "do", "das", "dos", "e"}


def tokens(nome):
    """'Kauã T. Brandão' → ['kaua', 't', 'brandao'] (sem acento, sem pontuação, sem partículas)."""
    sem = "".join(c for c in unicodedata.normalize("NFKD", nome) if not unicodedata.combining(c))
    limpo = "".join(c if c.isalnum() else " " for c in sem.casefold())
    return [t for t in limpo.split() if t not in PARTICULAS]


def pedaco_compativel(a, b):
    """Dois pedaços combinam se são iguais ou se um é a inicial do outro."""
    if a == b:
        return True
    if len(a) == 1 or len(b) == 1:
        return a[0] == b[0]
    return False


def compativel(nome1, nome2):
    """Mesmo número de partes, primeiro e último nome iguais por extenso, meio compatível.

    Exigir o sobrenome por extenso evita juntar "K. Brandão" com "Kátia Brandão" por acaso.
    """
    t1, t2 = tokens(nome1), tokens(nome2)
    if len(t1) != len(t2) or len(t1) < 2:
        return False
    if t1[-1] != t2[-1]:                       # sobrenome final precisa ser idêntico
        return False
    return all(pedaco_compativel(a, b) for a, b in zip(t1, t2))


# ── Testes rápidos (assert para se a regra quebrar) ───────────────────────
assert compativel("Kaua T Brandao", "Kauã Teixeira Brandão")
assert compativel("KAUÃ T. BRANDÃO", "kaua teixeira brandao")
assert not compativel("Kauã Brandão", "Kauã Teixeira Brandão")       # número de partes diferente
assert not compativel("M. R. Brandão", "Kauã Teixeira Brandão")
assert compativel("Maria da Silva", "MARIA SILVA")                    # partícula ignorada

# ── Coletar grafias de nomes nas evidências ───────────────────────────────
grafias = {}   # nome → de onde veio

rdap = json.loads((PASTA_CASO / "rdap_megaoferta.json").read_text(encoding="utf-8"))
for ent in rdap.get("entities", []):
    for campo in ent["vcardArray"][1]:
        if campo[0] == "fn":
            grafias[campo[3]] = "RDAP"

for linha in (PASTA_CASO / "whois_megaoferta.txt").read_text(encoding="utf-8").splitlines():
    if linha.startswith("owner:"):
        grafias.setdefault(linha.split(":", 1)[1].strip(), "WHOIS")

with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    for linha in csv.DictReader(arquivo, delimiter=";"):
        grafias.setdefault(linha["nome_vitima"], "denúncia")

grafias.setdefault(NOME_ALVO, "parâmetro")

# ── Agrupar: cada grupo junta grafias compatíveis entre si ────────────────
grupos = []
for nome in grafias:
    for grupo in grupos:
        if any(compativel(nome, outro) for outro in grupo):
            grupo.append(nome)
            break
    else:                                       # for...else: roda se não houve break
        grupos.append([nome])

print("(RDAP e WHOIS trazem a MESMA grafia, por isso ela aparece uma vez só.)")
print(f"🧾 {len(grafias)} grafias coletadas → {len(grupos)} pessoas prováveis\n")
for grupo in grupos:
    if len(grupo) > 1:
        print("🔗 Grupo com mais de uma grafia:")
        for nome in grupo:
            print(f"   • {nome:<28} ({grafias[nome]})")

alvo_grupo = next(g for g in grupos if NOME_ALVO in g)
print(f"\n🎯 Grafias compatíveis com {NOME_ALVO}: {len(alvo_grupo)}")

# Pares "quase iguais" que NÃO foram juntados (bom para revisão humana):
# mesmo primeiro nome E mesmo último sobrenome, mas a regra disse "não".
quase = [(a, b) for a, b in itertools.combinations(grafias, 2)
         if tokens(a)[0] == tokens(b)[0] and tokens(a)[-1] == tokens(b)[-1] and not compativel(a, b)]
print(f"🤔 Mesmo primeiro e último nome, mas NÃO compatíveis (revisar à mão): {len(quase)}")
for a, b in quase[:5]:
    print(f"   {a}  ×  {b}")

## 🔎 Inovação OSINT do dia

**Comparação "aproximada" de nomes com RapidFuzz.** Quando os nomes têm erros de digitação ("Brandao"
× "Brnadão"), a comparação exata falha. A biblioteca **RapidFuzz** (`pip install rapidfuzz`, licença
MIT) calcula semelhança entre textos muito rápido, com funções como `fuzz.ratio` e
`fuzz.token_sort_ratio` (que ignora a ordem das palavras: "Brandão Kauã" × "Kauã Brandão").
Combine com a chave sem acento desta carta: normalize primeiro, compare depois. Na biblioteca padrão,
`difflib.SequenceMatcher` faz algo parecido, só que mais devagar.

## 📚 Para ir além

- Documentação de `map`, `filter` e funções nativas: <https://docs.python.org/pt-br/3/library/functions.html>
- Métodos de string: <https://docs.python.org/pt-br/3/library/stdtypes.html#string-methods>
- `unicodedata` e formas de normalização: <https://docs.python.org/pt-br/3/library/unicodedata.html>
- chepy (citado pelo autor): <https://pypi.org/project/chepy/>
- RapidFuzz: <https://github.com/rapidfuzz/RapidFuzz>

## ✅ Checklist

- [ ] Rodei os cinco scripts originais e entendi o que o `map()` devolve
- [ ] Vi o `title()` errar em "Maria Da Silva" e consertei com `nome_br()`
- [ ] Transformei "Kauã Brandão" em "kaua brandao"
- [ ] Gerei a lista de variantes de username do suspeito
- [ ] Desmontei a conversa do WhatsApp em um CSV
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 16](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)